# Gemma Guide Demo Bootstrap

This notebook is the end-to-end bootstrap path for a fresh runtime.



## Token Setup

Paste your tokens into the cell below before running the clone and Hugging Face login steps.

- `GITHUB_TOKEN` is needed if the repo is private.
- `HF_TOKEN` is needed if model download requires authenticated Hugging Face access.

In [ ]:
import os

# Paste tokens here if they are not already set in the runtime environment.
GITHUB_TOKEN = ""
HF_TOKEN = ""

if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

print('GITHUB_TOKEN set:', bool(os.getenv('GITHUB_TOKEN')))
print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))

## Runtime Config

In [ ]:
from pathlib import Path
import os
import platform

REPO_URL = "https://github.com/pariidanDKE/GemmaGuide.git"
REPO_REF = "demo-setup"

if Path('/content').exists():
    REPO_DIR = Path('/content/GemmaGuide')
else:
    REPO_DIR = Path.cwd() / 'GemmaGuide'

HF_TOKEN = os.getenv('HF_TOKEN', '')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')

if GITHUB_TOKEN and REPO_URL.startswith('https://github.com/'):
    CLONE_URL = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
else:
    CLONE_URL = REPO_URL

APP_PORT = os.getenv('APP_PORT', '7862')
VLLM_PORT = os.getenv('VLLM_PORT', '8000')

print(f'Python: {platform.python_version()}')
print(f'Platform: {platform.platform()}')
print(f'Repo dir: {REPO_DIR}')
print(f'Repo ref: {REPO_REF}')
print(f'App port: {APP_PORT}')
print(f'vLLM port: {VLLM_PORT}')

## Serving Knobs

These are the main runtime knobs that affect whether the demo fits in memory and how much visual detail Gemma gets.

- `MODEL_CHOICE`: `e2b` is lighter and safer for tighter VRAM; `e4b` is stronger but heavier.
- `VLLM_QUANTIZATION`: use `fp8`, `bitsandbytes`, or `none`. `none` omits the quantization flag entirely.
- `MAX_SOFT_TOKENS`: visual token budget per image. Higher means more detail and more VRAM. Typical values: `140`, `280`, `560`.
- `MAX_MODEL_LEN`: context length. Lowering this is one of the cleanest VRAM reductions.
- `GPU_MEM_UTIL`: fraction of GPU memory vLLM is allowed to reserve. Too low can prevent startup.
- `MAX_NUM_SEQS`: keep this at `1` on smaller GPUs.
- `TIPS_SHORT_SIDE`: lower values reduce TIPS memory and latency.
- `VLLM_EXTRA_ARGS`: optional raw extra flags appended to the `vllm serve` command.

In [ ]:
MODEL_CHOICE = 'e2b'
MODEL_OPTIONS = {
    'e2b': ('google/gemma-4-E2B-it', 'gemma-4-e2b-it'),
    'e4b': ('google/gemma-4-E4B-it', 'gemma-4-e4b-it'),
}

VLLM_MODEL_REPO, VLLM_SERVED_NAME = MODEL_OPTIONS[MODEL_CHOICE]
VLLM_QUANTIZATION = 'fp8'
MAX_SOFT_TOKENS = '280'
MAX_MODEL_LEN = '4096'
GPU_MEM_UTIL = '0.60'
MAX_NUM_SEQS = '1'
TENSOR_PARALLEL = '1'
TIPS_SHORT_SIDE = '672'
VLLM_EXTRA_ARGS = ''

print('model repo =', VLLM_MODEL_REPO)
print('served name =', VLLM_SERVED_NAME)
print('quantization =', VLLM_QUANTIZATION)
print('max_soft_tokens =', MAX_SOFT_TOKENS)
print('max_model_len =', MAX_MODEL_LEN)
print('gpu_mem_util =', GPU_MEM_UTIL)
print('max_num_seqs =', MAX_NUM_SEQS)
print('tensor_parallel =', TENSOR_PARALLEL)
print('tips_short_side =', TIPS_SHORT_SIDE)
print('vllm_extra_args =', VLLM_EXTRA_ARGS or '<none>')

## Clone The Repo

If the repo already exists in the runtime, this cell leaves it in place.

In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', CLONE_URL, str(REPO_DIR)], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], check=True)

## Install Dependencies

In [ ]:
%pip install --upgrade pip
%pip install -r {REPO_DIR / 'requirements.txt'}

In [ ]:
import shutil
import subprocess

if shutil.which('ffmpeg') is None:
    print('ffmpeg not found; attempting install...')
    subprocess.run(['bash', '-lc', 'apt-get update -y && apt-get install -y ffmpeg'], check=True)
else:
    print('ffmpeg already available')

## Hugging Face Login

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged into Hugging Face using HF_TOKEN from the environment.')
else:
    print('HF_TOKEN is empty. If model download fails, set HF_TOKEN and rerun this cell.')

## Start The Demo Stack

This starts `vllm`, the app, and a Cloudflare Quick Tunnel in the background, then prints structured status as JSON.

The model is started as a separate server process on purpose. That keeps the app modular, preserves a clean OpenAI-compatible seam, and makes it easier to swap `vllm` for another backend later without rewriting the app flow.

Logs are written to `/tmp/gemma_demo_logs/`:
- `vllm.log`
- `app.log`
- `cloudflared.log`
- `demo_state.json`

In [ ]:
import subprocess
import sys

start_cmd = [
    sys.executable,
    'scripts/demo_bootstrap.py',
    'start',
    '--with-tunnel',
    '--app-port', APP_PORT,
    '--vllm-port', VLLM_PORT,
    '--model', VLLM_MODEL_REPO,
    '--vllm-served-name', VLLM_SERVED_NAME,
    '--quantization', VLLM_QUANTIZATION,
    '--max-soft-tokens', MAX_SOFT_TOKENS,
    '--max-model-len', MAX_MODEL_LEN,
    '--gpu-mem-util', GPU_MEM_UTIL,
    '--max-num-seqs', MAX_NUM_SEQS,
    '--tensor-parallel', TENSOR_PARALLEL,
    '--tips-short-side', TIPS_SHORT_SIDE,
]
if VLLM_EXTRA_ARGS:
    start_cmd.extend(['--vllm-extra-args', VLLM_EXTRA_ARGS])
subprocess.run(start_cmd, cwd=REPO_DIR, check=True)

## Inspect Status

In [ ]:
import json
import subprocess
import sys

status_raw = subprocess.check_output(
    [sys.executable, 'scripts/demo_bootstrap.py', 'status'],
    cwd=REPO_DIR,
    text=True,
)
status = json.loads(status_raw)
print(json.dumps(status, indent=2))
print('\nOpen this URL on your phone:')
print(status.get('urls', {}).get('public'))

## Log Helpers

In [ ]:
from pathlib import Path
import json

state_path = Path('/tmp/gemma_demo_logs/demo_state.json')
state = json.loads(state_path.read_text()) if state_path.exists() else {}

def tail_file(path, lines=80):
    path = Path(path)
    if not path.exists():
        print(f'{path} does not exist')
        return
    content = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    print('\n'.join(content[-lines:]))

for name, info in state.get('services', {}).items():
    print(f'=== {name} :: {info.get("log_path")} ===')
    tail_file(info['log_path'], lines=40)
    print()

## Cleanup

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/demo_bootstrap.py', 'stop'], cwd=REPO_DIR, check=True)